# DataFrame Thinking: SQL Semantics, NULLs, Partition Pruning, Lazy Plans

## What you'll practice

Spark is a SQL engine with a Python API. Thinking in its terms — set-based queries, three-valued NULL logic, partition pruning, and lazy plans that an optimizer rewrites — matters more than
memorizing functions. The last exercise builds a tiny lazy DataFrame with filter and projection pushdown, so `df.explain()` will feel familiar.

> Function and class names in **Implement** are what the tests look for. Run all tests for this notebook with `pytest tests/pyspark_prep/test_12_03_dataframe_thinking.py -v` from the lab root. Untouched exercises show as *skipped*.
>
> Solutions: `solutions/12_pyspark_preparation/03_dataframe_thinking.ipynb`. Don't open it until you've tried.

## Exercise 1 — 🟢 Easy · A SQL query in plain Python

**Implement:** `country_report(orders: list[dict]) -> list[tuple[str, int, float]]`

Implement exactly: `SELECT country, COUNT(*) AS n, ROUND(SUM(amount), 2) AS total FROM orders WHERE status <> 'CANCELLED' GROUP BY country HAVING COUNT(*) >= 2 ORDER BY total DESC, country`.
Note SQL semantics: a NULL status makes `status <> 'CANCELLED'` unknown → the row is filtered out.

**Input:** `orders: list[dict]`

**Expected output:** `list[tuple[str, int, float]]`

**Constraints:** Standard library only (the last notebook uses the PySpark DataFrame API).

**Example:** the tests are executable examples — read `test_country_report` in `tests/pyspark_prep/test_12_03_dataframe_thinking.py`.

**Check:** `pytest tests/pyspark_prep/test_12_03_dataframe_thinking.py -k test_country_report`

**YOUR CODE ↓**

## Exercise 2 — 🟡 Medium · The same query in pandas

**Implement:** `country_report_pd(df: pd.DataFrame) -> pd.DataFrame`

Same semantics (including NULL status filtered out) returning a DataFrame with columns `country, n, total` and a fresh index. Use `groupby(...).agg(n=..., total=...)`, `.query`/boolean masks and `sort_values`.

**Input:** `df: pd.DataFrame`

**Expected output:** `pd.DataFrame`

**Constraints:** Standard library only (the last notebook uses the PySpark DataFrame API).

**Example:** the tests are executable examples — read `test_country_report_pd` in `tests/pyspark_prep/test_12_03_dataframe_thinking.py`.

**Check:** `pytest tests/pyspark_prep/test_12_03_dataframe_thinking.py -k test_country_report_pd`

**YOUR CODE ↓**

## Exercise 3 — 🟡 Medium · Three-valued logic and NULL-aware aggregates

**Implement:** `sql_eq(a, b)`, `sql_and(a, b)`, `sql_or(a, b)`, `sql_count(values, star: bool = False) -> int`, `sql_avg(values)`

Implement SQL semantics where `None` means NULL/UNKNOWN: `sql_eq` returns `None` if either side is NULL, else bool. `sql_and`/`sql_or` follow Kleene logic (`False AND NULL = False`,
`True OR NULL = True`, otherwise NULL propagates). `sql_count(values)` counts non-NULLs; `sql_count(values, star=True)` counts rows. `sql_avg` ignores NULLs and returns `None` for no non-NULL values
(so `AVG` ≠ `SUM/COUNT(*)` when NULLs exist).

**Input:** `a, b`

**Constraints:** Standard library only (the last notebook uses the PySpark DataFrame API).

**Example:** the tests are executable examples — read `test_three_valued_logic, test_null_aggregates` in `tests/pyspark_prep/test_12_03_dataframe_thinking.py`.

**Check:** `pytest tests/pyspark_prep/test_12_03_dataframe_thinking.py -k "test_three_valued_logic or test_null_aggregates"`

**YOUR CODE ↓**

## Exercise 4 — 🟠 Hard · Partition pruning

**Implement:** `prune(paths: list[str], filters: dict[str, tuple[str, object]]) -> tuple[list[str], int]`

Given Hive-style file paths (`sales/dt=2026-01-03/country=IN/part-0.parquet`) and filters on partition columns — operators `"="`, `"!="`, `">="`, `"<="`, `">"`, `"<"`, `"in"` (value is a set) — return
`(paths_to_read, files_skipped)`. Paths missing a filtered partition column are **kept** (can't prove they're irrelevant). Values compare as strings (works for ISO dates). Unknown operator → `ValueError`.

**Input:** `paths: list[str], filters: dict[str, tuple[str, object]]`

**Expected output:** `tuple[list[str], int]`

**Constraints:** Standard library only (the last notebook uses the PySpark DataFrame API).

**Example:** the tests are executable examples — read `test_prune` in `tests/pyspark_prep/test_12_03_dataframe_thinking.py`.

**Check:** `pytest tests/pyspark_prep/test_12_03_dataframe_thinking.py -k test_prune`

**YOUR CODE ↓**

## Challenge — 🔴 Very Hard · A lazy DataFrame with pushdown and explain() · 🧩 Mini-project

**Implement:** `class LazyFrame`

`LazyFrame.read_csv(path)` creates a lazy plan; `.filter(column, op, value)`, `.with_column(name, func, depends_on=[cols])`, `.select(*cols)` append plan nodes and return new frames;
nothing is read until `.collect()` (→ list of dicts). Before executing, **optimize**:
1. *Filter pushdown*: filters on columns that come straight from the source (not created by `with_column`) move to the scan, applied while reading rows.
2. *Projection pushdown*: compute the set of source columns actually needed (selected columns + filter columns + `depends_on` of used derived columns) and read only those (drop others at scan time).
`.explain()` returns a string with one line per optimized node, starting with `Scan csv [needed cols] filters=[...]`. Track `frame.last_scan_stats = {"rows_read", "rows_after_scan_filters", "columns_read"}`.
Values from CSV are strings; `filter` converts the compared value with `type(value)` of the literal you pass (`filter("quantity", ">", 1)` compares as int).

**Input:** see the class description above

**Expected output:** an instance behaving as described

**Constraints:** Standard library only (the last notebook uses the PySpark DataFrame API).

**Example:** the tests are executable examples — read `test_lazy_frame, test_lazy_frame_no_select_reads_all` in `tests/pyspark_prep/test_12_03_dataframe_thinking.py`.

**Check:** `pytest tests/pyspark_prep/test_12_03_dataframe_thinking.py -k "test_lazy_frame or test_lazy_frame_no_select_reads_all"`

**YOUR CODE ↓**